# Ingest - many tables in ONE job task

One `lakefed_ingest` run (a *sweep*) processes a whole **task_collection** of (tenant db, table)
tasks in this single notebook, using a thread pool + Spark **FAIR scheduler pools** so each
table's work overlaps on the cluster. This decouples throughput from the workspace job limits
(2,000 concurrent task runs; 10,000 run-submits/hour) that a task-per-table design hits - the
throttle becomes compute (cluster cores), which is the intended cost model. Shard by
`task_collection` for horizontal scale: run one sweep per collection.

Notes:
- **One concurrency knob, `parallelism`** = the most units of query work running at once in this
  run, of ANY type: a CT read+MERGE, a non-partitioned seed, ONE partition of a partitioned seed, a
  small source query (CT version / min-valid / size / bounds), or the staging->target swap. Each unit
  holds one slot of a shared `parallel.WorkSlots(parallelism)`; threads may outnumber slots, but a
  thread only runs Spark/JDBC work while holding a slot, and never holds one while waiting (e.g. a
  table waiting for its partitions) - so nested fan-out can't deadlock. Each slot is its own FAIR pool
  (`pool0..pool{parallelism-1}`). The budget is per run: concurrent sweeps of different collections
  on separate job clusters each have their own.
- **Skip-no-change:** per source database, the CT version is captured once and tables are checked for
  changes in batched queries; only changed tables are read and merged. Unchanged tables are bulk-advanced
  in Lakebase (checkpoint + freshness) — no reads, no MERGEs, no empty Delta commits.
- Cluster must run the **FAIR** scheduler (`spark.scheduler.mode=FAIR`) for the pools to apply.
- **Strategies and sources:** each control row's `load_type` selects a load strategy
  (`load_strategies.py`: `ct` and `full` today; `cursor` / `cdc` are placeholders) and its `src_type` a
  source adapter (`source_adapters.py`: SQL Server, Oracle, PostgreSQL, MySQL). A row whose pair is not
  supported fails on its own (stage `plan`) with a clear message. `full` rows reload every cycle.
- **Write modes:** `write_mode` 'merge' (default) keeps each sink in step with its source table.
  'append' (`ct` and `full`) appends instead: every CT change (`__change_op` I / U / D, `__change_version`)
  and every seed or full-load snapshot (`__change_op` 'S') becomes new rows, stamped `__extracted_at` and
  `__run_id` - a change log many source databases can fan in to (`client_id`, plus `client_attrs`
  identity columns), for downstream processing such as SCD2.
- All source-touching work is in **ct_engine.py** (Spark JDBC; no SQL warehouse, no UC connection) -
  this notebook is pure orchestration + Lakebase checkpointing + telemetry.
- One source read per table (per range for partitioned seeds), no `.cache()`: the seed streams the
  JDBC read into the Delta write and the CT increment feeds the CHANGETABLE read straight into the
  MERGE. Partitioned seeds load a staging table and swap it into the target in one atomic commit, so
  readers never see a partial table.
- Because the per-table work runs in-process (no Jobs-UI task per table), progress/telemetry is
  written to Lakebase: one `ingest_run` row per run and one `ingest_event` row per (run, table),
  queryable live and read directly from the Lakebase UC catalog (see docs/monitoring.md).
- Per-table failures are isolated (one bad table does not fail the collection) and recorded to
  both `ingest_event` (with the stage it failed at) and the `checkpoint` status for triage.

In [ ]:
# Cluster task. Bootstrap: put this notebook's dir on sys.path and import the shared modules.
import sys, os, json

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_dir = os.path.dirname(ctx.notebookPath().get())
if ("/Workspace" + nb_dir) not in sys.path:
    sys.path.append("/Workspace" + nb_dir)

import state_store as ss
from state_store import StateStore
import ct_engine
import cursor_engine
import load_strategies
import parallel

In [ ]:
# Params + connect to Lakebase (SDK-minted OAuth token; no stored password), then open a run.
def widget(name, default=""):
    dbutils.widgets.text(name, default)
    return dbutils.widgets.get(name)


task_collection = widget("task_collection", "example_cohort")
db_instance = widget("db_instance", "lakefed-ingest")
db_name = widget("db_name", "lakefed_ingest")
# Max concurrent units of query work in this run (all types share it; see the header).
parallelism = int(widget("parallelism", "32"))
# Skip-no-change: tables per batched change-detection statement (one source query per chunk).
detection_chunk = int(widget("detection_chunk", str(ct_engine.DEFAULT_DETECTION_CHUNK)))
# Source credentials: per-DB JSON secrets in this scope (Databricks- or Azure Key Vault-backed);
# each control row names its own secret_key. TLS trust is configurable (prod: "false").
secret_scope = widget("secret_scope", "lakefed_ingest")
trust_server_certificate = widget("trust_server_certificate", "true")
# Change reads under SNAPSHOT isolation (needs ALLOW_SNAPSHOT_ISOLATION ON on every source database).
snapshot_isolation = widget("snapshot_isolation", "false")
# Off: a table whose select_list gained columns carries on from its checkpoint instead of reseeding
# (the writes add the columns; rows written before stay NULL in them), e.g. an append sink.
reseed_on_new_columns = widget("reseed_on_new_columns", "true").strip().lower() in ("1", "true", "yes")
# Append mode: up to this many tables appending to the same consolidated sink are read in one Spark
# job and written in one Delta commit (batch_append.py). 1 = one table at a time.
append_batch_size = int(widget("append_batch_size", "1"))
# Onboarding: at most this many seeds per run (the rest wait for later runs). 0 = no limit.
max_seeds_per_run = int(widget("max_seeds_per_run", "0"))
# Unchanged tables already at the captured version are written to Lakebase only when their
# last_success_at is older than this many seconds. 0 = stamp every unchanged table every run.
freshness_interval_s = float(widget("freshness_interval_s", "0"))
# Concurrent queries per source server, across every run sharing this Lakebase database: each run
# takes its share, this divided by the runs reading the server now, at least 1. 0 = no cap.
max_queries_per_source = int(widget("max_queries_per_source", "0"))
os.environ["LAKEFED_SQL_SECRET_SCOPE"] = secret_scope
os.environ["LAKEFED_SQL_TRUST_SERVER_CERT"] = trust_server_certificate
os.environ["LAKEFED_SQL_SNAPSHOT_ISOLATION"] = snapshot_isolation
# Single source read per table: Delta MERGE materializes a non-Delta (JDBC) source itself when
# this is "auto" (the default) - we add no .cache() of our own. Logged once for visibility.
# (spark.conf.get validates a supplied default against the allowed values, so read it bare.)
try:
    _materialize = spark.conf.get("spark.databricks.delta.merge.materializeSource")
except Exception:
    _materialize = "auto (unset; runtime default)"
print("spark.databricks.delta.merge.materializeSource =", _materialize)

# reconnect: a dropped state connection is reopened (fresh token) instead of failing the run.
store = StateStore.connect_lakebase(db_instance, db_name, reconnect=True)


def _safe(fn, default=""):
    try:
        return fn() or default
    except Exception:
        return default


job_run_id = _safe(lambda: str(ctx.jobRunId().get()))
cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId", "")
# Per-collection run lock: different task_collections sweep concurrently; a second run of the SAME
# collection (e.g. a schedule firing while the previous sweep is still going) exits as 'skipped'.
# Session-level advisory lock on a DEDICATED connection held for the whole sweep — released
# automatically if the connection drops, so a crashed run never leaves a stale lock.
lock_store = StateStore.connect_lakebase(
    db_instance, db_name, application_name=ss.lock_application_name(task_collection, job_run_id))
if not lock_store.try_lock_collection(task_collection):
    lock_store.close()
    skipped_id = store.start_run(task_collection, job_run_id, cluster_id, parallelism)
    store.finish_run(skipped_id, total=0, ok=0, failed=0, status="skipped",
                       detail="another sweep of this task_collection is already running")
    print(f"run {skipped_id}: SKIPPED - task_collection {task_collection} is already being swept")
    dbutils.notebook.exit(json.dumps({"run_id": skipped_id, "task_collection": task_collection,
                                      "status": "skipped"}))

configs = store.list_task_configs(task_collection)

# Lost-lock guard: if the lock connection drops mid-sweep (network blip, Lakebase restart), the lock
# is released and another run could start. Workers re-check it at most every 30s and stop STARTING
# new tables once it is gone; skipped tables are simply picked up by the next sweep.
import threading, time
_lock_state = {"checked": time.monotonic(), "held": True}
_lock_mu = threading.Lock()


def assert_lock_held(every_s=30):
    with _lock_mu:
        if _lock_state["held"] and time.monotonic() - _lock_state["checked"] >= every_s:
            _lock_state["checked"] = time.monotonic()
            _lock_state["held"] = lock_store.is_alive()
        if not _lock_state["held"]:
            raise RuntimeError("collection lock lost; table deferred to the next sweep")
run_id = store.start_run(task_collection, job_run_id, cluster_id, parallelism)
print(f"run {run_id}: collection {task_collection} -> {len(configs)} tables (parallelism={parallelism})")
# This run holds the collection's lock, so the collection's other open runs are dead (cancelled, or
# their cluster went away): close them and their in-flight table events, which would otherwise stay
# 'running' in the monitoring views forever. Telemetry only: a failure here never stops the sweep.
try:
    closed_runs, closed_events = store.close_abandoned_runs(task_collection, run_id)
    if closed_runs:
        print(f"closed {closed_runs} earlier run(s) of {task_collection} that ended without finishing "
              f"({closed_events} table events left running)")
except Exception as e:
    print(f"[telemetry] could not close earlier unfinished runs: {str(e)[:200]}")

In [ ]:
# Per-(tenant db, table) work with Lakebase telemetry. CT/source ops live in ct_engine (Spark JDBC);
# everything here is orchestration + checkpointing + telemetry, keyed by the control id (cfg["id"]).
# Each table arrives with a ROUTE decided by the planning phase (next cell):
#   full             - load_type 'full': reload the whole table (data-only atomic replace; staged
#                      swap when partitioned), checkpoint {strategy full, phase loaded, no position}
#   seed             - no checkpoint / phase 'seeding': capture V0, seed, checkpoint {V0, incremental}
#   schema_reseed    - control.select_list gained columns vs the recorded one: enqueue a reseed
#   retention_reseed - batched detection found the checkpoint below CT's min valid version
#   merge            - batched detection found changes: CHANGETABLE read + MERGE, advance to `cur`
#                      (the database version captured BEFORE detection; no per-table version queries)
#   fallback         - detection for this table's chunk failed: the full per-table path
#   cursor_seed      - load_type 'cursor', no usable checkpoint: capture the high watermark `hi`,
#                      seed, checkpoint {cursor position hi, incremental}
#   cursor           - load_type 'cursor': read `hi`; unchanged if equal to the checkpoint, else read
#                      (position, hi] + upsert MERGE by PK, advance to `hi`
#   pending          - write_mode 'append': a run stopped between an append and its checkpoint advance.
#                      The sink holds that run's rows -> advance (recovered); else seed / fallback
# Append mode (control.write_mode 'append') appends where merge mode merges or replaces: seeds and full
# loads as snapshot rows (operation 'S'), CT changes with their operation and version. Before a CT
# append, the position it reaches and this run are recorded on the checkpoint (pending), so a run
# that stops before the advance is detected next run instead of appending the same changes twice.
# Unchanged CT tables never reach this worker (bulk-advanced in one Lakebase update).
# The `stage` variable records how far a table got, so a failure pinpoints read vs merge vs
# checkpoint. The collection lock is re-checked (cached, <=1 query/30s) right before every commit
# point — seed, each partition + the swap (via `guard`), MERGE, reseed enqueue, checkpoint advance —
# so a run that lost its lock stops writing; skipped work is redone by the next sweep (at-least-once).
from functools import partial
from concurrent.futures import ThreadPoolExecutor

_ACTION = {"full": "full", "seed": "seed", "merge": "increment", "fallback": "increment",
           "schema_reseed": "reseed", "retention_reseed": "reseed",
           "cursor_seed": "seed", "cursor": "increment", "pending": "increment", "recovered": "increment"}


_RESEEDED = object()


def _apply_changes_or_reseed(cfg, sink, v_from, v_to, run_id):
    """Read one table's changes up to ``v_to`` (the version the checkpoint advances to) and MERGE
    them, or append them (append mode, after recording the pending append). If CT retention passed
    the checkpoint after planning, the read-time guard in ct_engine makes SQL Server raise; queue a
    reseed for that table instead of failing it (and never advance its checkpoint)."""
    try:
        if ct_engine.is_append(cfg):
            store.record_pending(cfg, v_to, run_id)
            return ct_engine.read_and_append_ct(spark, cfg, sink, v_from, v_to, slots=slots, run_id=run_id)
        return ct_engine.read_and_merge_ct(spark, cfg, sink, v_from, slots=slots, to_version=v_to)
    except Exception as e:
        if not ct_engine.is_retention_lapsed_error(e):
            raise
        assert_lock_held()
        store.enqueue_reseed(cfg, reason="retention_miss")
        print(f"[reseed:retention_miss@read] {cfg['src_database']}.{cfg['src_schema']}.{cfg['src_table']}")
        return _RESEEDED


def ingest_one(plan, run_id):
    assert_lock_held()
    cfg, route, cp = plan["cfg"], plan["route"], plan.get("cp")
    db, schema, table = cfg["src_database"], cfg["src_schema"], cfg["src_table"]
    sink = ct_engine.sink_fqn(cfg)
    pool = ""  # FAIR pools follow work slots (one per unit); the ones used are recorded at finish
    slots.start_tracking()
    phase = cp["phase"] if cp else ss.SEEDING
    v_from = planner.position_from(plan)
    action = _ACTION[route]
    event_id = store.record_event_start(run_id, cfg, action, phase, pool)

    stage = "plan"
    rows_read = rows_merged = v_to = None
    try:
        if route == "pending":   # append mode: did the stopped run's append commit?
            stage = "verify"
            committed = ct_engine.append_committed(spark, cfg, sink, cp["pending_run_id"], slots=slots)
            route = "recovered" if committed else ("fallback" if phase == ss.INCREMENTAL else "seed")
            action = "seed" if phase != ss.INCREMENTAL else _ACTION[route]
        if route == "recovered":
            stage = "checkpoint"
            assert_lock_held()
            v_to = int(cp["pending_position"])
            store.advance_checkpoint(cfg, position=v_to, detail="pending append committed",
                                     select_list=cfg["select_list"])
            if phase != ss.INCREMENTAL:   # the append was the seed: record CT's begin_version as a seed does
                store.record_ct_begin_version(cfg, ct_engine.ct_begin_version(cfg, slots=slots))
            outcome = "recovered"
        elif route == "full":
            stage = "load"
            assert_lock_held()
            loaded = load_strategies.get_strategy("full").load(
                spark, cfg, sink, slots=slots, executor=unit_executor, guard=assert_lock_held, run_id=run_id)
            rows_merged = loaded if isinstance(loaded, int) else None
            stage = "checkpoint"
            assert_lock_held()
            store.advance_checkpoint(cfg, position=None, phase=ss.LOADED, detail="full load complete",
                                     select_list=cfg["select_list"])
            outcome = "loaded"
        elif route == "seed":
            stage = "ensure_sink"
            ct_engine.ensure_sink_table(spark, cfg, sink, slots=slots)
            stage = "read"
            # V0 and the table's CT begin_version in ONE query, BEFORE the seed; raises unless CT is on
            # for the database and the table (never seed a table CT can't follow).
            v0, begin_version = ct_engine.prepare_seed(spark, cfg, slots=slots)
            stage = "seed"
            assert_lock_held()
            if ct_engine.is_append(cfg):
                store.record_pending(cfg, v0, run_id)
            seeded = ct_engine.seed(spark, cfg, sink, slots=slots, executor=unit_executor,
                                    guard=assert_lock_held, run_id=run_id, version=v0)
            rows_merged = seeded if isinstance(seeded, int) else None
            stage = "checkpoint"
            assert_lock_held()
            store.advance_checkpoint(cfg, position=v0, phase=ss.INCREMENTAL, detail="seed complete",
                                     select_list=cfg["select_list"])
            store.record_ct_begin_version(cfg, begin_version)
            v_to = v0
            outcome = "seeded"
        elif route == "cursor_seed":
            stage = "ensure_sink"
            ct_engine.ensure_sink_table(spark, cfg, sink, slots=slots)
            stage = "read"
            typ = cursor_engine.column_type(cfg, slots=slots)
            hi = cursor_engine.high_watermark(cfg, typ, slots=slots)  # capture BEFORE the seed
            stage = "seed"
            assert_lock_held()
            seeded = ct_engine.seed(spark, cfg, sink, slots=slots, executor=unit_executor,
                                    guard=assert_lock_held)
            rows_merged = seeded if isinstance(seeded, int) else None
            stage = "checkpoint"
            assert_lock_held()
            store.advance_checkpoint(cfg, position=cursor_engine.encode_position(cfg["cursor_column"], typ, hi),
                                     phase=ss.INCREMENTAL, detail="seed complete", select_list=cfg["select_list"])
            v_to = hi
            outcome = "seeded"
        elif route == "cursor":
            pos = plan["position"]
            stage = "read"
            hi = cursor_engine.high_watermark(cfg, pos["type"], slots=slots)  # upper bound BEFORE the read
            decision, new_value = cursor_engine.plan_increment(pos, hi)
            if decision == "read":
                stage = "ensure_sink"
                ct_engine.ensure_sink_table(spark, cfg, sink, slots=slots)  # no-op when the sink exists
                stage = "merge"
                assert_lock_held()
                merged = cursor_engine.read_and_merge(spark, cfg, sink, pos["type"], pos["value"], hi,
                                                      slots=slots)
                rows_merged = merged if isinstance(merged, int) else None
                outcome = "incremented"
            else:
                outcome = "unchanged"
            stage = "checkpoint"
            assert_lock_held()
            store.advance_checkpoint(cfg, position=cursor_engine.encode_position(pos["col"], pos["type"], new_value),
                                     select_list=cfg["select_list"])
            v_to = new_value
        elif route in ("schema_reseed", "retention_reseed"):
            stage = "checkpoint"
            assert_lock_held()
            reason = plan.get("reason") or ("schema_change" if route == "schema_reseed" else "retention_miss")
            store.enqueue_reseed(cfg, reason=reason)  # flips phase -> seeding
            print(f"[reseed:{reason}] {db}.{schema}.{table}")
            outcome = "reseed_enqueued"
        elif route == "merge":
            stage = "ensure_sink"
            ct_engine.ensure_sink_table(spark, cfg, sink, slots=slots)  # no-op when the sink exists
            stage = "merge"
            assert_lock_held()
            merged = _apply_changes_or_reseed(cfg, sink, v_from, plan["cur"], run_id)
            if merged is _RESEEDED:
                outcome, action = "reseed_enqueued", "reseed"
            else:
                rows_merged = merged if isinstance(merged, int) else None
                stage = "checkpoint"
                assert_lock_held()
                store.advance_checkpoint(cfg, position=plan["cur"], select_list=cfg["select_list"])
                if plan.get("ct_begin") is not None and (cp or {}).get("ct_begin_version") is None:
                    store.record_ct_begin_version(cfg, plan["ct_begin"])   # legacy checkpoint
                v_to = plan["cur"]
                outcome = "incremented"
        else:  # fallback: the per-table path (this table's batched detection query failed)
            stage = "ensure_sink"
            ct_engine.ensure_sink_table(spark, cfg, sink, slots=slots)
            stage = "read"
            new_cols = (ct_engine.missing_sink_columns(cfg["select_list"], spark.table(sink).columns)
                        if reseed_on_new_columns else [])
            mv = None if new_cols else ct_engine.min_valid_version(cfg, slots=slots)
            if new_cols or ss.decide_action(phase, v_from, mv) == "reseed":
                stage = "checkpoint"
                assert_lock_held()
                reason = "schema_change" if new_cols else "retention_miss"
                store.enqueue_reseed(cfg, reason=reason)
                print(f"[reseed:{reason}] {db}.{schema}.{table} (fallback)")
                outcome = "reseed_enqueued"
                action = "reseed"
            else:
                new_version = ct_engine.current_version(cfg, slots=slots)  # upper bound BEFORE the read
                if new_version is None:
                    raise RuntimeError(ct_engine.CT_NOT_ENABLED_MSG)
                stage = "merge"
                assert_lock_held()
                merged = _apply_changes_or_reseed(cfg, sink, v_from, new_version, run_id)
                if merged is _RESEEDED:
                    outcome, action = "reseed_enqueued", "reseed"
                else:
                    rows_merged = merged if isinstance(merged, int) else None
                    stage = "checkpoint"
                    assert_lock_held()
                    store.advance_checkpoint(cfg, position=new_version, select_list=cfg["select_list"])
                    v_to = new_version
                    outcome = "incremented"

    except Exception as e:
        err = str(e)
        pool = _pools_label(cfg, action)
        try:
            store.record_event_finish(event_id, "failed", stage=stage, rows_read=rows_read,
                                      rows_merged=rows_merged, position_from=v_from,
                                      position_to=v_to, error=err, pool=pool)
        except Exception:
            pass
        try:
            store.set_status(cfg, "failed", err[:500])
        except Exception:
            pass
        print(f"[failed@{stage}] {db}.{schema}.{table} pool={pool}: {err[:200]}")
        raise

    # Success-path telemetry runs AFTER the try: the ingest (and its checkpoint) already committed,
    # so a telemetry write error must not flip this table to failed — log it and move on.
    # A reseed copied no data: record it as 'skipped' (action 'reseed') so last_success_at is NOT
    # stamped and the freshness/lag view keeps showing the table as stale until it re-seeds.
    pool = _pools_label(cfg, action)
    try:
        store.record_event_finish(event_id, "skipped" if action == "reseed" else "ok", stage=stage,
                                  rows_read=rows_read, rows_merged=rows_merged,
                                  position_from=v_from, position_to=v_to, action=action, pool=pool)
    except Exception as te:
        print(f"[telemetry] {db}.{schema}.{table}: record_event_finish failed (ingest unaffected): {str(te)[:200]}")
    print(f"[ok:{outcome}] {db}.{schema}.{table} pool={pool}")
    return outcome


def _pools_label(cfg, action):
    """FAIR pools this table's own units ran in (e.g. 'pool3,pool17'). A partitioned seed's partitions
    run on the shared executor in whatever pools are free, so they're summarized, not listed."""
    used = ",".join(slots.tracked_pools()) or "-"
    return used + ("+partitions" if action in ("seed", "full") and cfg.get("load_partitioned") else "")


worker = partial(ingest_one, run_id=run_id)


def set_pool(pool):
    # Called when a unit acquires a slot: that unit's Spark jobs run in the slot's FAIR pool.
    spark.sparkContext.setLocalProperty("spark.scheduler.pool", pool)


# ONE budget for all query work in this run, plus a shared executor that runs partition units.
# Table workers (run_parallel below) only orchestrate: they hold a slot just while doing a unit
# themselves and never while waiting on their partitions, so slots can't deadlock.
# max_queries_per_source: this run's share of each server's cap, recounted every 15 s from the runs
# reading the server (shared advisory locks held on the lock connection, released with it).
source_limiter, source_keys = None, {}
if max_queries_per_source > 0:
    source_keys = lock_store.join_sources({ct_engine.source_key(c) for c in configs})
    source_limiter = parallel.SourceLimiter()

    def refresh_source_shares():
        runs = lock_store.source_runs(source_keys.values())
        for source, share in ss.source_shares(max_queries_per_source, source_keys, runs).items():
            source_limiter.set_limit(source, share)

    try:
        refresh_source_shares()
    except Exception as e:   # the other runs can't be counted: cap this run alone at the whole limit
        print(f"[caps] could not count the runs per source ({str(e)[:200]}); this run takes the whole cap")
        for source in source_keys:
            source_limiter.set_limit(source, max_queries_per_source)

    def _refresh_source_shares_loop():
        while True:
            time.sleep(15)
            try:
                refresh_source_shares()
            except Exception as e:
                print(f"[caps] share refresh failed, keeping the last shares: {str(e)[:200]}")

    threading.Thread(target=_refresh_source_shares_loop, daemon=True, name="source-shares").start()
    print("source caps (this run's share):", {s: source_limiter.limit(s) for s in source_keys})
slots = parallel.WorkSlots(parallelism, set_pool=set_pool, sources=source_limiter)
unit_executor = ThreadPoolExecutor(max_workers=parallelism, thread_name_prefix="unit")

In [ ]:
# Planning (src/lakefed_ingest/planner.py — pure, unit-tested), then the per-table fan-out.
#  0. Sink ownership is checked (invalid client_id; a sink shared by a dedicated row and consolidated
#     rows), then each row's (src_type, load_type) against the support matrix: unsupported pairs and
#     placeholder strategies fail per table (stage 'plan'); 'full' rows go straight to a reload;
#     'cursor' rows go to the worker (cursor_seed / cursor: one high-watermark query per table).
#  1. ONE Lakebase query loads every checkpoint; each CT table is routed (seed / schema_reseed /
#     pending append / detection candidate). Legacy checkpoints without a recorded select_list check
#     the sink once. With reseed_on_new_columns off, added columns don't route to schema_reseed.
#  2. Candidates are grouped by source DATABASE (CT versions are per database, never comparable across
#     databases). Per database: ONE query captures cur = CHANGE_TRACKING_CURRENT_VERSION() and every
#     table's CT state; if every checkpoint >= cur nothing changed (fast exit); else batched detection
#     (chunks of `detection_chunk` tables, one query each) classifies each table changed / unchanged / reseed.
#     A failed chunk falls back to the per-table path. A database that fails entirely (unreachable,
#     auth) has its tables recorded as failed (stage 'detect') — the other databases continue.
#     Correctness: `cur` is captured BEFORE detection, so any change with version <= cur is visible to
#     the EXISTS check; later changes carry higher versions and are picked up next cycle from the
#     advanced checkpoint (at-least-once; the MERGE is idempotent).
#  3. Unchanged tables: ONE bulk Lakebase update advances each table's checkpoint to ITS database's
#     `cur` (never backwards) and stamps freshness — no source reads, no MERGEs, no empty commits.
#     With freshness_interval_s, tables whose checkpoint wouldn't move are stamped once per interval.
#  4. Only seeds (at most max_seeds_per_run), reseeds, changed tables and fallbacks go through the
#     per-table worker or, for append sinks with append_batch_size > 1, through batches (batch_append.py).
import time as _time
from concurrent.futures import ThreadPoolExecutor as _TPE
import planner
import batch_append

t_plan = _time.monotonic()
checkpoints = store.get_checkpoints([c["id"] for c in configs])
plans, candidates, legacy = planner.route_tables(configs, checkpoints, mixed_sinks=store.mixed_sinks(),
                                                 reseed_on_new_columns=reseed_on_new_columns,
                                                 snapshot_isolation=ct_engine.snapshot_isolation())
unsupported = [p for p in plans if p["route"] == "unsupported"]
plans = [p for p in plans if p["route"] != "unsupported"]
if unsupported:   # e.g. a placeholder (cdc; cursor on non-SQL Server) or 'ct' on a non-SQL Server source
    store.record_failed_tables(run_id, [(p["cfg"], p["cp"], p["error"]) for p in unsupported], stage="plan")
    for p in unsupported[:5]:
        c = p["cfg"]
        print(f"[failed@plan] {c['src_database']}.{c['src_schema']}.{c['src_table']}: {p['error']}")
for p in plans:
    if p["route"] == "schema_reseed":
        c = p["cfg"]
        print(f"[schema] {c['src_database']}.{c['src_schema']}.{c['src_table']}: select_list added {p['added']}")

# Legacy checkpoints (no recorded select_list): compare with the sink's columns ONCE (catalog
# lookups, run in parallel), then record the select_list on the advance so later cycles compare
# in memory.
record_sl = set()
def _legacy_check(item):
    cfg, cp = item
    sink = ct_engine.sink_fqn(cfg)
    try:
        return item, ct_engine.missing_sink_columns(cfg["select_list"], spark.table(sink).columns)
    except Exception:
        return item, None          # sink missing/unreadable -> per-table fallback path
if legacy:
    with _TPE(max_workers=max(1, min(parallelism, 32))) as ex:
        for (cfg, cp), new_cols in ex.map(_legacy_check, legacy):
            if new_cols is None:
                plans.append({"cfg": cfg, "route": "fallback", "cp": cp})
            elif new_cols and reseed_on_new_columns:
                plans.append({"cfg": cfg, "route": "schema_reseed", "cp": cp, "added": new_cols})
            else:
                record_sl.add(int(cfg["id"]))
                candidates.append((cfg, cp))

groups = planner.group_by_source(candidates)
group_members = list(groups.values())
t_detect = _time.monotonic()
group_results = parallel.run_parallel(
    group_members,
    lambda members: planner.detect_group(
        members,
        current_version=None,
        database_state=lambda c: ct_engine.database_ct_state(c, slots=slots),
        detect=lambda c, items: ct_engine.detect_changes(spark, c, items, slots=slots,
                                                         chunk_size=detection_chunk),
        record_sl=record_sl),
    max(1, min(len(groups) or 1, parallelism)), set_pool=None,
    item_id=lambda m: m[0][0].get("src_database"))
outputs = []
for members, r in zip(group_members, group_results):   # run_parallel preserves input order
    if r["status"] == "failed":   # unexpected planner error: isolate that group, never drop its tables
        outputs.append({"database": r["item"], "cur": None, "plans": [], "unchanged": [],
                        "failed": [(c, cp, f"change detection error: {r['detail']}") for c, cp in members],
                        "queries": 0, "fast_exit": False, "fallback": 0})
    else:
        outputs.append(r["detail"])
agg = planner.merge_group_results(outputs)
plans.extend(agg["plans"])
plans, deferred_seeds = planner.apply_seed_budget(plans, max_seeds_per_run)   # the rest seed in later runs
detection = {"databases": len(groups), "queries": agg["queries"], "fast_exits": agg["fast_exits"],
             "fallback_tables": agg["fallback_tables"], "failed_databases": agg["failed_databases"],
             "group_versions": agg["group_versions"], "seeds_deferred": len(deferred_seeds),
             "detect_s": round(_time.monotonic() - t_detect, 1)}

if agg["failed"]:   # whole-database failures: visible per table, other databases unaffected
    store.record_failed_tables(run_id, agg["failed"], stage="detect")
    for c, _, err in agg["failed"][:5]:
        print(f"[failed@detect] {c['src_database']}.{c['src_schema']}.{c['src_table']}: {err[:200]}")

t_bulk = _time.monotonic()
to_write = planner.bulk_rows_to_write(agg["unchanged"], checkpoints, freshness_interval_s)
if to_write:
    assert_lock_held()
    store.bulk_advance_unchanged(to_write)   # each row carries ITS database's cur
detection["bulk_advance_rows"] = len(to_write)
detection["bulk_advance_ms"] = int((_time.monotonic() - t_bulk) * 1000)
detection["plan_s"] = round(_time.monotonic() - t_plan, 1)
detection["routes"] = {k: sum(1 for p in plans if p["route"] == k)
                       for k in ("full", "seed", "schema_reseed", "retention_reseed", "merge", "fallback",
                                 "cursor_seed", "cursor", "pending")}
detection["unsupported"] = len(unsupported)
detection["unchanged"] = len(agg["unchanged"])
# Append sinks with append_batch_size > 1: changed tables, seeds and full loads appending to the same
# consolidated sink run in batches, one Spark job + one Delta commit each (batch_append.py).
batches, plans = batch_append.plan_batches(plans, append_batch_size, max_reads=parallelism)
detection["batches"] = batch_append.describe(batches)
print("plan:", json.dumps(detection, default=str))

# Per-table work (seeds, reseeds, changed tables, fallbacks, cursor tables) and batches: every unit
# draws on `slots` (a batch holds one slot per table it reads at once).
batcher = batch_append.BatchRunner(spark, store, slots, run_id, per_table=worker, guard=assert_lock_held)
work = batch_append.order_work(batches, plans)
results = batch_append.flatten(
    parallel.run_parallel(work, lambda item: batcher.run(item[1]) if item[0] == "batch" else worker(item[1]),
                          parallelism, set_pool=None, item_id=batch_append.work_id),
    work)
unit_executor.shutdown(wait=True)
summary = parallel.summarize(results)
n_unchanged, n_detect_failed, n_unsupported = len(agg["unchanged"]), len(agg["failed"]), len(unsupported)
summary["total"] += n_unchanged + n_detect_failed + n_unsupported
summary["ok"] += n_unchanged
summary["failed"] += n_detect_failed + n_unsupported
if n_unchanged:   # add to the worker's own 'unchanged' (cursor tables whose watermark didn't move)
    summary["by_outcome"]["unchanged"] = summary["by_outcome"].get("unchanged", 0) + n_unchanged
if n_detect_failed:
    summary["by_outcome"]["failed_detect"] = n_detect_failed
if n_unsupported:
    summary["by_outcome"]["failed_plan"] = n_unsupported
by = summary.get("by_outcome", {})
# Per source: this run's last share of max_queries_per_source and the most queries it ran at once.
source_caps = {s: {"share": source_limiter.limit(s), "peak": source_limiter.peak.get(s, 0)}
               for s in source_keys} if source_limiter is not None else {}
reseeded = by.get("reseed_enqueued", 0)
store.finish_run(
    run_id,
    total=summary["total"], ok=summary["ok"], failed=summary["failed"],
    skipped=by.get("unchanged", 0), reseeded=reseeded,
    status="completed" if summary["failed"] == 0 else "failed",
    detail=json.dumps({"parallelism": parallelism, "peak_units": slots.peak, "detection": detection,
                       "source_caps": source_caps}, default=str),
)

summary["run_id"] = run_id
summary["task_collection"] = task_collection
summary["parallelism"] = parallelism
summary["peak_units"] = slots.peak
summary["source_caps"] = source_caps
summary["detection"] = detection
print(json.dumps(summary, indent=2, default=str))

failures = [r for r in results if r["status"] == "failed"]
if failures:
    print("failures:", json.dumps(failures, default=str)[:2000])

lock_store.close()  # closing the session releases the collection lock (safe even if it already dropped)

dbutils.notebook.exit(json.dumps(summary, default=str))